# Pydantic model to structure output

In [ ]:
# TODO:skapa en agent som ska simulera en IT-anställd
from pydantic_ai import Agent
from dotenv import load_dotenv

load_dotenv()

employee_simulator_agent = Agent(
    "openrouter:liquid/lfm-2.5-1.2b-thinking:free",
    system_prompt="""
    You are an HR expert within IT field in Sweden within data science, data engineering,
machine learning, AI engineering. You will simulate IT employees.

Fields to include in output:
- name
- age
- gender
- job_title
- salary in SEK per month
    """
)



result = await employee_simulator_agent.run("simulate two employees")

result

AgentRunResult(output='Here are two simulated employee profiles as per your request:\n\n**Employee 1:**  \n- **Name:** Erik Malle  \n- **Age:** 35  \n- **Gender:** Male  \n- **Job Title:** Data Engineer  \n- **Salary:** 480,000 SEK/month  \n\n**Employee 2:**  \n- **Name:** Anna Lund  \n- **Age:** 28  \n- **Gender:** Female  \n- **Job Title:** Machine Learning Specialist  \n- **Salary:** 450,000 SEK/month  \n\nThese roles align with Sweden’s IT/data science landscape, reflecting diverse professional contributions. Let me know if you need adjustments!')

In [4]:
print(result.output)

Here are two simulated employee profiles as per your request:

**Employee 1:**  
- **Name:** Erik Malle  
- **Age:** 35  
- **Gender:** Male  
- **Job Title:** Data Engineer  
- **Salary:** 480,000 SEK/month  

**Employee 2:**  
- **Name:** Anna Lund  
- **Age:** 28  
- **Gender:** Female  
- **Job Title:** Machine Learning Specialist  
- **Salary:** 450,000 SEK/month  

These roles align with Sweden’s IT/data science landscape, reflecting diverse professional contributions. Let me know if you need adjustments!


In [5]:
with open('simulated_employees.md', 'w') as file:
    file.write(result.output)

## Get more structured output

issue with above:

 - output structure vary    
 - hard to work with the data e.g. compute mean of salaries

 want:
    - repeatable structure

In [1]:
from pydantic import BaseModel, Field
from typing import Literal
from pydantic_ai import Agent

class employeeModel(BaseModel):
    name: str = Field(description='Mostly swedish names, but could be foreign names as well')
    age: int = Field(description= 'age should be between 18 and 67')
    gender: Literal['Male','Female']
    experience_level: Literal['Entry', 'Mid level', 'Senior', 'Expert']
    job_title: str
    salary: int = Field(gte=30_000, lte=50_000, description='salary shouldb be between 30k and 50k, the higher experience level, the higher salary')

employee_simulator_agent = Agent(
    "openrouter:nvidia/nemotron-nano-12b-v2-vl:free",
    system_prompt="""
    You are an HR expert within IT field in Sweden within data science, data engineering,
machine learning, AI engineering. You will simulate IT employees.""", retries=1
)

result= await employee_simulator_agent.run('Give me three employees', output_type=list[employeeModel])
result

C:\Users\Lilit\AppData\Local\Temp\ipykernel_24672\3097466144.py:11: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'gte', 'lte'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  salary: int = Field(gte=30_000, lte=50_000, description='salary shouldb be between 30k and 50k, the higher experience level, the higher salary')


AgentRunResult(output=[employeeModel(name='Anna Johansson', age=32, gender='Female', experience_level='Mid level', job_title='Data Scientist', salary=40000), employeeModel(name='Marcus Karlsson', age=38, gender='Male', experience_level='Senior', job_title='Software Engineer', salary=45000), employeeModel(name='Lena Svensson', age=45, gender='Female', experience_level='Expert', job_title='AI Specialist', salary=48000)])

In [2]:
result.output[0]

employeeModel(name='Anna Johansson', age=32, gender='Female', experience_level='Mid level', job_title='Data Scientist', salary=40000)

In [3]:
# BaseModel -> dictionary
result.output[0].model_dump()

{'name': 'Anna Johansson',
 'age': 32,
 'gender': 'Female',
 'experience_level': 'Mid level',
 'job_title': 'Data Scientist',
 'salary': 40000}

### TODO:
- result.output make into list of dictionaries
- create pandas dataframe based on the list 

In [4]:
employees = result.output
employees

[employeeModel(name='Anna Johansson', age=32, gender='Female', experience_level='Mid level', job_title='Data Scientist', salary=40000),
 employeeModel(name='Marcus Karlsson', age=38, gender='Male', experience_level='Senior', job_title='Software Engineer', salary=45000),
 employeeModel(name='Lena Svensson', age=45, gender='Female', experience_level='Expert', job_title='AI Specialist', salary=48000)]

In [5]:
employees_list = [employee.model_dump() for employee in employees]
employees_list

[{'name': 'Anna Johansson',
  'age': 32,
  'gender': 'Female',
  'experience_level': 'Mid level',
  'job_title': 'Data Scientist',
  'salary': 40000},
 {'name': 'Marcus Karlsson',
  'age': 38,
  'gender': 'Male',
  'experience_level': 'Senior',
  'job_title': 'Software Engineer',
  'salary': 45000},
 {'name': 'Lena Svensson',
  'age': 45,
  'gender': 'Female',
  'experience_level': 'Expert',
  'job_title': 'AI Specialist',
  'salary': 48000}]

In [7]:
import pandas as pd 

df = pd.DataFrame(employees_list)
df

,name,age,gender,experience_level,job_title,salary
0,Anna Johansson,32,Female,Mid level,Data Scientist,40000
1,Marcus Karlsson,38,Male,Senior,Software Engineer,45000
2,Lena Svensson,45,Female,Expert,AI Specialist,48000


In [8]:
df['salary'].mean()

np.float64(44333.333333333336)

In [9]:
df.to_csv('simulated_employees.csv', index=False)

Exercise:

### MLOps Engineer
select the following fields
- education_name
- program_description (summary)
- career_options (list of jobs)
- course_list
- length

### Contact oss
 select 
- department
- opening_hours
- phone_time
- phone

export 2 json-files

## Import data

In [10]:
with open('mlops.txt', 'r') as file:
    mlops_text = file.read()

mlops_text[:150]

'Vill du jobba i skÃ¤rningspunkten mellan AI, data och teknik? Som MLOps Engineer lÃ¤r du dig att bygga, driftsÃ¤tta och optimera maskininlÃ¤rningsmode'

## Data models

In [11]:
from pydantic import BaseModel, Field

class Education(BaseModel):
    education_name: str = Field(description='Make sure to find this information in the text')
    program_description: str = Field(description='A 3-5 sentences summary about the program')
    career_options: list[str] 
    courses: list[str]
    length: float = Field(description='Total study length of the program')